# Paimon INTERNAL via Spark notebook

This notebook registers and validates a PAIMON INTERNAL catalog against Kasanari.

In [1]:
import json
import requests

base_url = "http://kasanari:9090"
catalog_id = "paimon_spark_internal"

payload = {
    "catalogId": catalog_id,
    "catalogType": "PAIMON",
    "mode": "INTERNAL",
    "spec": {
        "fileIoProperties": {
            "fs.s3a.access.key": "admin",
            "fs.s3a.secret.key": "password",
            "fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
            "fs.s3a.path.style.access": "true",
            "fs.s3a.endpoint": "http://minio:9000",
        },
        "catalogProperties": {
            "warehouse": "s3a://warehouse",
            "uri": "jdbc:postgresql://catalog-storage:5432/postgres",
            "kasanari.jdbc.user": "postgres",
            "kasanari.jdbc.password": "postgres",
            "kasanari.catalog.key": catalog_id,
        },
    },
}

response = requests.post(f"{base_url}/management/v1/catalogs", json=payload, timeout=20)
print(response.status_code)
print(response.text)


201
{"catalogId":"paimon_spark_internal","catalogType":"PAIMON","mode":"INTERNAL","spec":{"fileIoProperties":{"fs.s3a.access.key":"admin","fs.s3a.secret.key":"password","fs.s3a.impl":"org.apache.hadoop.fs.s3a.S3AFileSystem","fs.s3a.path.style.access":"true","fs.s3a.endpoint":"http://minio:9000"},"catalogProperties":{"warehouse":"s3a://warehouse","uri":"jdbc:postgresql://catalog-storage:5432/postgres","kasanari.jdbc.user":"postgres","kasanari.jdbc.password":"postgres","kasanari.catalog.key":"paimon_spark_internal"}},"version":1}


In [2]:
response = requests.get(f"{base_url}/management/v1/catalogs/PAIMON/{catalog_id}", timeout=20)
print(response.status_code)
print(json.dumps(response.json(), indent=2))

200
{
  "catalogId": "paimon_spark_internal",
  "catalogType": "PAIMON",
  "mode": "INTERNAL",
  "spec": {
    "fileIoProperties": {
      "fs.s3a.access.key": "admin",
      "fs.s3a.secret.key": "password",
      "fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
      "fs.s3a.path.style.access": "true",
      "fs.s3a.endpoint": "http://minio:9000"
    },
    "catalogProperties": {
      "warehouse": "s3a://warehouse",
      "uri": "jdbc:postgresql://catalog-storage:5432/postgres",
      "kasanari.jdbc.user": "postgres",
      "kasanari.jdbc.password": "postgres",
      "kasanari.catalog.key": "paimon_spark_internal"
    }
  },
  "version": 1
}


## Spark SQL operations through Paimon REST catalog

This section demonstrates create/insert/select/alter/view/delete/drop operations via Spark SQL.

In [3]:
import uuid
from pyspark.sql import SparkSession

spark_catalog = "kasanari_paimon"

spark = (
    SparkSession.builder
    .appName("kasanari-paimon-internal-ops")
    .master("local[*]")
    .config("spark.jars", "/home/jovyan/extra-jars/paimon-spark-runtime-4_2.13-1.4.1.jar")
    .config("spark.sql.extensions", "org.apache.paimon.spark.extensions.PaimonSparkSessionExtensions")
    .config(f"spark.sql.catalog.{spark_catalog}", "org.apache.paimon.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{spark_catalog}.metastore", "rest")
    .config(f"spark.sql.catalog.{spark_catalog}.uri", "http://kasanari:9090/paimon")
    .config(f"spark.sql.catalog.{spark_catalog}.warehouse", catalog_id)
    .config(f"spark.sql.catalog.{spark_catalog}.token.provider", "bear")
    .config(f"spark.sql.catalog.{spark_catalog}.token", "token")
    .config(f"spark.sql.catalog.{spark_catalog}.rest.client.content-type", "application/json")
    .config(f"spark.sql.catalog.{spark_catalog}.header.content-type", "application/json")
    .config(f"spark.sql.catalog.{spark_catalog}.header.Content-Type", "application/json")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)

db = "demo"
table = f"events_{uuid.uuid4().hex[:8]}"
view = f"{table}_v"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {spark_catalog}.{db}")
spark.sql(
    f"""
    CREATE TABLE {spark_catalog}.{db}.{table} (
      id INT,
      event STRING,
      source STRING
    ) TBLPROPERTIES ('primary-key' = 'id', 'bucket'='1')
    """
)

spark.sql(
    f"""
    INSERT INTO {spark_catalog}.{db}.{table}
    VALUES
      (1, 'signup', 'spark'),
      (2, 'click', 'spark'),
      (3, 'purchase', 'spark')
    """
)

print("Initial rows:")
spark.sql(f"SELECT * FROM {spark_catalog}.{db}.{table} ORDER BY id").show(truncate=False)

spark.sql(f"ALTER TABLE {spark_catalog}.{db}.{table} ADD COLUMNS (notes STRING)")
spark.sql(f"UPDATE {spark_catalog}.{db}.{table} SET notes = 'ok' WHERE id IN (1, 2)")

spark.sql(
    f"CREATE OR REPLACE VIEW {spark_catalog}.{db}.{view} AS "
    f"SELECT id, event FROM {spark_catalog}.{db}.{table} WHERE id <= 2"
)

print("View rows:")
spark.sql(f"SELECT * FROM {spark_catalog}.{db}.{view} ORDER BY id").show(truncate=False)

spark.sql(f"DELETE FROM {spark_catalog}.{db}.{table} WHERE id = 3")

print("After delete:")
spark.sql(f"SELECT id, event, notes FROM {spark_catalog}.{db}.{table} ORDER BY id").show(truncate=False)

spark.sql(f"DROP VIEW {spark_catalog}.{db}.{view}")
spark.sql(f"DROP TABLE {spark_catalog}.{db}.{table}")

print("Done: created, inserted, selected, altered, viewed, deleted, and dropped objects.")

Py4JJavaError: An error occurred while calling o61.sql.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 2.0 failed 1 times, most recent failure: Lost task 0.0 in stage 2.0 (TID 3) (78b8555e9a32 executor driver): java.lang.NoClassDefFoundError: com/amazonaws/AmazonClientException
	at java.base/java.lang.Class.forName0(Native Method)
	at java.base/java.lang.Class.forName(Class.java:469)
	at org.apache.hadoop.conf.Configuration.getClassByNameOrNull(Configuration.java:2674)
	at org.apache.hadoop.conf.Configuration.getClassByName(Configuration.java:2639)
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2735)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3569)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3612)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:172)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3716)
	at org.apache.hadoop.fs.FileSystem$